# ClearFlow — Phase 04: Feature Selection

**Goal:** rank the 70 model features using three methods, then define the planned 10 feature sets.

1. **Random Forest importance** — impurity-based feature importance.
2. **ANOVA F-test** — a univariate statistical ranking of numeric features. This is used instead of chi-square because the flow features need not be non-negative.
3. **RFE (Decision Tree)** — recursively removes less important features and ranks features by how long they survive.

Each method produces a full ranking of all 70 features. The top 10, top 20, and top 40 form nine sets. The tenth set is the shared **all 70 features** baseline.

## Leakage and sampling policy

- Read the **training split only**. The test split must not affect feature selection.
- Use the same deterministic, class-capped reservoir sample for all three selectors (up to 10,000 rows per class, maximum 120,000 rows).
- Smaller classes are retained in full. Sampling does not change the Phase 3 train/test files and is not model-training class balancing.
- The selection sample and settings are recorded in the Phase 04 manifest.

The capped sample is a deliberate computational trade-off. When interpreting rankings, remember that they are estimated from this training-only sample rather than every training row.

In [1]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "data" / "processed" / "train.csv.gz").is_file():
    parent = PROJECT_ROOT.parent
    if (parent / "data" / "processed" / "train.csv.gz").is_file():
        PROJECT_ROOT = parent

if not (PROJECT_ROOT / "data" / "processed" / "train.csv.gz").is_file():
    raise FileNotFoundError(
        "Could not find data/processed/train.csv.gz. Open this notebook from the ClearFlow project "
        "and complete Phase 03 first."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.feature_selection import run_phase04_feature_selection

print("Project root:", PROJECT_ROOT)
print("Training split:", PROJECT_ROOT / "data" / "processed" / "train.csv.gz")

Project root: C:\Users\arhat\OneDrive\Desktop\Amrita\sem5\DV\Project\ClearFlow
Training split: C:\Users\arhat\OneDrive\Desktop\Amrita\sem5\DV\Project\ClearFlow\data\processed\train.csv.gz


## 1. Run the feature selection pipeline

The full training file is scanned in chunks. The selection sample is kept in memory, not the entire training split. This cell fits the three methods and writes the ranking and feature-set exports.

In [2]:
results = run_phase04_feature_selection(
    project_root=PROJECT_ROOT,
    chunk_size=100_000,
    max_rows_per_class=10_000,
    random_state=42,
    rfe_step=0.20,
)

print("Export directory:", results["output_dir"])

PASS 1/2: reading the training split and building a class-capped sample...
  Training rows scanned: 2,020,546
  Feature-selection sample: 70,245 rows x 70 features
  Maximum sample size: 10,000 rows per class
Fitting selector 1/3: Random Forest importance...
Fitting selector 2/3: ANOVA F-test...


C:\Users\arhat\OneDrive\Desktop\Amrita\sem5\DV\Project\ClearFlow\.venv\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:112: UserWarning: Features [31 32 48] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)


Fitting selector 3/3: recursive feature elimination (Decision Tree)...

Feature-set summary:
feature_set_id  feature_count               selection_method_display
      rf_top10             10               Random Forest Importance
      rf_top20             20               Random Forest Importance
      rf_top40             40               Random Forest Importance
   anova_top10             10                           ANOVA F-test
   anova_top20             20                           ANOVA F-test
   anova_top40             40                           ANOVA F-test
     rfe_top10             10                    RFE (Decision Tree)
     rfe_top20             20                    RFE (Decision Tree)
     rfe_top40             40                    RFE (Decision Tree)
  all_features             70 Shared baseline (no feature selection)

PASS: wrote Phase 4 exports to C:\Users\arhat\OneDrive\Desktop\Amrita\sem5\DV\Project\ClearFlow\exports\phase04_feature_selection
Export directory:

## 2. Selection-sample composition

Use this table to see how many training rows were scanned and how many rows from each class were used for ranking.

In [3]:
display(results["sample_summary"].sort_values("Label").reset_index(drop=True))

,Label,train_rows,selection_sample_rows,sample_pct_of_class,max_rows_per_class,cap_reached
0,Benign,1677531,10000,0.596114,10000,True
1,Bot,1580,1580,100.000000,10000,False
2,DDoS,102474,10000,9.758573,10000,True
3,DoS GoldenEye,8191,8191,100.000000,10000,False
4,DoS Hulk,138293,10000,7.231024,10000,True
5,DoS Slowhttptest,4168,4168,100.000000,10000,False
6,DoS slowloris,4317,4317,100.000000,10000,False
7,FTP-Patator,4751,4751,100.000000,10000,False
8,PortScan,72003,10000,13.888310,10000,True
9,SSH-Patator,2614,2614,100.000000,10000,False


## 3. Top-ranked features from each method

Scores have **different meanings across methods**. Compare rankings and feature membership, not the raw score values across methods.

In [4]:
for method in ["rf_importance", "anova_f_test", "rfe_decision_tree"]:
    print("\n", results["rankings"].loc[results["rankings"]["method"] == method, "method_display"].iloc[0])
    display(
        results["rankings"]
        .loc[results["rankings"]["method"] == method]
        .sort_values("rank")
        .head(20)[["rank", "feature", "score", "score_name", "p_value", "elimination_round"]]
        .reset_index(drop=True)
    )


 Random Forest Importance


,rank,feature,score,score_name,p_value,elimination_round
0,1,Destination Port,0.068724,gini_importance,NaN,NaN
1,2,Init_Win_bytes_backward,0.055316,gini_importance,NaN,NaN
2,3,min_seg_size_forward,0.042184,gini_importance,NaN,NaN
3,4,Init_Win_bytes_forward,0.040060,gini_importance,NaN,NaN
4,5,Bwd Packets/s,0.033274,gini_importance,NaN,NaN
5,6,Bwd Header Length,0.027688,gini_importance,NaN,NaN
6,7,Fwd Packet Length Max,0.026208,gini_importance,NaN,NaN
7,8,Subflow Bwd Packets,0.021546,gini_importance,NaN,NaN
8,9,Flow IAT Mean,0.021256,gini_importance,NaN,NaN
9,10,Packet Length Mean,0.020557,gini_importance,NaN,NaN



 ANOVA F-test


,rank,feature,score,score_name,p_value,elimination_round
0,1,Init_Win_bytes_backward,23171.118166,anova_f_statistic,0.0,NaN
1,2,Bwd Packet Length Mean,8666.378504,anova_f_statistic,0.0,NaN
2,3,Avg Bwd Segment Size,8666.378504,anova_f_statistic,0.0,NaN
3,4,Fwd IAT Std,8325.872186,anova_f_statistic,0.0,NaN
4,5,Packet Length Mean,7817.096753,anova_f_statistic,0.0,NaN
5,6,Average Packet Size,7544.719396,anova_f_statistic,0.0,NaN
6,7,Bwd Packet Length Max,7141.539411,anova_f_statistic,0.0,NaN
7,8,Packet Length Std,7111.685383,anova_f_statistic,0.0,NaN
8,9,SYN Flag Count,6958.897582,anova_f_statistic,0.0,NaN
9,10,Max Packet Length,6699.361209,anova_f_statistic,0.0,NaN



 RFE (Decision Tree)


,rank,feature,score,score_name,p_value,elimination_round
0,1,Destination Port,0.539428,decision_tree_importance_at_elimination,NaN,15.0
1,2,Flow IAT Std,0.460572,decision_tree_importance_at_elimination,NaN,14.0
2,3,min_seg_size_forward,0.255994,decision_tree_importance_at_elimination,NaN,13.0
3,4,Fwd Packet Length Max,0.185175,decision_tree_importance_at_elimination,NaN,12.0
4,5,Init_Win_bytes_backward,0.143059,decision_tree_importance_at_elimination,NaN,11.0
5,6,Bwd Packet Length Max,0.118627,decision_tree_importance_at_elimination,NaN,11.0
6,7,Average Packet Size,0.073468,decision_tree_importance_at_elimination,NaN,10.0
7,8,Flow Bytes/s,0.070499,decision_tree_importance_at_elimination,NaN,10.0
8,9,Bwd IAT Mean,0.049947,decision_tree_importance_at_elimination,NaN,9.0
9,10,Active Min,0.046222,decision_tree_importance_at_elimination,NaN,9.0


## 4. Verify the 10 feature sets

Expected: nine sets with 10, 20, or 40 features, plus one shared set containing all 70 features.

In [5]:
display(results["feature_set_summary"])

assert len(results["feature_set_summary"]) == 10
assert results["feature_set_summary"]["feature_count"].tolist() == [10, 20, 40, 10, 20, 40, 10, 20, 40, 70]
assert len(results["rankings"]) == 3 * 70
assert results["manifest"]["test_split_used_for_selection"] is False
print("PASS: 3 complete rankings, 10 feature sets, and no test-split use.")

,set_order,feature_set_id,feature_set_name,selection_method,selection_method_display,requested_feature_count,feature_count
0,1,rf_top10,Top 10 — Random Forest Importance,rf_importance,Random Forest Importance,10,10
1,2,rf_top20,Top 20 — Random Forest Importance,rf_importance,Random Forest Importance,20,20
2,3,rf_top40,Top 40 — Random Forest Importance,rf_importance,Random Forest Importance,40,40
3,4,anova_top10,Top 10 — ANOVA F-test,anova_f_test,ANOVA F-test,10,10
4,5,anova_top20,Top 20 — ANOVA F-test,anova_f_test,ANOVA F-test,20,20
5,6,anova_top40,Top 40 — ANOVA F-test,anova_f_test,ANOVA F-test,40,40
6,7,rfe_top10,Top 10 — RFE (Decision Tree),rfe_decision_tree,RFE (Decision Tree),10,10
7,8,rfe_top20,Top 20 — RFE (Decision Tree),rfe_decision_tree,RFE (Decision Tree),20,20
8,9,rfe_top40,Top 40 — RFE (Decision Tree),rfe_decision_tree,RFE (Decision Tree),40,40
9,10,all_features,All 70 Features (Shared Baseline),shared_all_features,Shared baseline (no feature selection),70,70


PASS: 3 complete rankings, 10 feature sets, and no test-split use.


## 5. Inspect feature-set overlap

`jaccard_similarity` ranges from 0 (no shared features) to 1 (identical sets). The full pairwise table is exported for Tableau/Power BI.

In [6]:
display(
    results["overlap"]
    .loc[results["overlap"]["feature_set_a"] != results["overlap"]["feature_set_b"]]
    .sort_values(["jaccard_similarity", "intersection_count"], ascending=False)
    .head(20)
    .reset_index(drop=True)
)

,feature_set_a,feature_set_a_name,feature_set_b,feature_set_b_name,features_in_a,features_in_b,intersection_count,union_count,jaccard_similarity
0,rf_top40,Top 40 — Random Forest Importance,rfe_top40,Top 40 — RFE (Decision Tree),40,40,31,49,0.632653
1,rfe_top40,Top 40 — RFE (Decision Tree),rf_top40,Top 40 — Random Forest Importance,40,40,31,49,0.632653
2,rf_top40,Top 40 — Random Forest Importance,all_features,All 70 Features (Shared Baseline),40,70,40,70,0.571429
3,anova_top40,Top 40 — ANOVA F-test,all_features,All 70 Features (Shared Baseline),40,70,40,70,0.571429
4,rfe_top40,Top 40 — RFE (Decision Tree),all_features,All 70 Features (Shared Baseline),40,70,40,70,0.571429
5,all_features,All 70 Features (Shared Baseline),rf_top40,Top 40 — Random Forest Importance,70,40,40,70,0.571429
6,all_features,All 70 Features (Shared Baseline),anova_top40,Top 40 — ANOVA F-test,70,40,40,70,0.571429
7,all_features,All 70 Features (Shared Baseline),rfe_top40,Top 40 — RFE (Decision Tree),70,40,40,70,0.571429
8,rf_top20,Top 20 — Random Forest Importance,rf_top40,Top 40 — Random Forest Importance,20,40,20,40,0.500000
9,rf_top40,Top 40 — Random Forest Importance,rf_top20,Top 20 — Random Forest Importance,40,20,20,40,0.500000


## 6. Export inventory

The output files are in `exports/phase04_feature_selection/`:

- `feature_rankings_long.csv` — 210 rows, three methods × 70 features.
- `feature_rankings_wide.csv` — one row per feature with method ranks/scores.
- `feature_sets_long.csv` — feature membership for each of the 10 sets.
- `feature_set_summary.csv` — the 10 set definitions.
- `feature_set_overlap.csv` — pairwise intersection counts and Jaccard similarity.
- `selection_sample_summary.csv` — class counts for the training-only selection sample.
- `feature_sets.json` — ordered feature lists to be reused during model training.
- `feature_selection_manifest.json` and `README.txt` — parameters, caveats and export descriptions.

**Next phase:** use these fixed feature lists for the four models and five seeds. Do not redo feature selection per model or seed, and never refit feature selection on the test data.